In [ ]:
import os
import glob
import tqdm
import torch
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from torch import nn
from torch import optim
from sklearn.metrics import f1_score, precision_score, recall_score
from torch.utils.data import DataLoader

from utils.configs import TrainingConfig, KFDTrainingConfig
from models.sunet import SUNet2EnResConv
from torchmetrics.classification import BinaryF1Score
from models.cnn import L3CNNV4

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"Available device is {device}")

Available device is cuda:0


### Key Frame Detection Model

In [17]:
@torch.no_grad()
def evaluate_key_frame_detection_model(model: nn.Module, dataloader: DataLoader, loss_fn, device: torch.device):
    y_pred = []
    y_true = []
    loss_values = []

    for event_frames, scores in tqdm.tqdm(dataloader):
        event_frames = event_frames.to(dtype=torch.float32, device=device)
        scores = scores.to(dtype=torch.float32, device=device)
        
        pred_logits = model(event_frames)
        pred_scores = torch.sigmoid(pred_logits)
        
        loss = loss_fn(pred_logits, scores)
        
        y_pred.extend(pred_scores.squeeze(1).cpu().numpy().tolist())
        y_true.extend(scores.squeeze(1).cpu().numpy().tolist())
        loss_values.append(loss.item())
        
    y_pred = np.array(y_pred)
    y_true = np.array(y_true)
    loss = np.mean(loss_values)
 
    metrics_dict = {
        "f1_score": f1_score(y_true, np.where(y_pred > 0.4, 1, 0), average='binary'),
        "precision": precision_score(y_true, np.where(y_pred > 0.4, 1, 0), average='binary'),
        "recall": recall_score(y_true, np.where(y_pred > 0.4, 1, 0), average='binary'),
        "loss": loss
    }
    
    return metrics_dict

In [14]:
model_dir = "checkpoints/L3CNNV4_20250908_040454"

config = KFDTrainingConfig.load(f"{model_dir}/config.pkl")

model = L3CNNV4(in_channels=1, down_sample_factor=config.downsample_factor, input_size=config.input_size).to(device)
model.load_state_dict(torch.load(f"{model_dir}/best_loss.pth", weights_only=True, map_location=device))
model = model.eval()

In [18]:
dataset_dir = "data/raw_data/keyframe_detection"
output_dir = f"results/keyframe_detection/{os.path.basename(model_dir)}"

os.makedirs(output_dir, exist_ok=True)

npz_files = [f for f in glob.glob(os.path.join(dataset_dir, "*.npz"))]

rows = []

for npz_file in tqdm.tqdm(npz_files):
    output_file = os.path.join(output_dir, os.path.basename(npz_file).replace(".npz", ".csv"))
    
    if os.path.exists(output_file):
        continue
    
    data = np.load(npz_file)
    
    event_frames = data["frames"]
    key_frames = data["key_frame"]
    ego_velocities = data["ego_velocity"]
    shortest_distances = data["shortest_distance"]
    new_actors = data["num_new_actors"]

    rows = []

    with torch.no_grad():
        for i in range(event_frames.shape[0]):
            event_frame = torch.tensor(event_frames[i], dtype=torch.float32, device=device).unsqueeze(0).unsqueeze(0)
            
            pred_logits = model(event_frame)
            pred_score = torch.sigmoid(pred_logits).item()

            rows.append({
                "ego_velocity": ego_velocities[i],
                "shortest_distance": shortest_distances[i],
                "num_new_actors": new_actors[i],
                "key_frame": key_frames[i],
                "pred_score": pred_score
            })
            
    results = pd.DataFrame(rows)
    results.to_csv(output_file, index=False)

100%|██████████| 164/164 [15:14<00:00,  5.58s/it]


In [23]:
train_sequences = np.loadtxt("data/slicing/train_sequences.txt", dtype=str).tolist()
test_sequences = np.loadtxt("data/slicing/test_sequences.txt", dtype=str).tolist()

results_dir = "results/keyframe_detection/L3CNNV4_20250908_040454"
results = {f.replace(".csv", ""): pd.read_csv(os.path.join(results_dir, f)) for f in os.listdir(results_dir) if f.endswith(".csv")}

In [24]:
rows = []

for sequence, df in results.items():
    split = "train" if sequence in train_sequences else "test" if sequence in test_sequences else "unknown"
    
    town = sequence.split("_")[0]
    
    df.loc[:, "split"] = split
    df.loc[:, "town"] = town
    
    rows.append(df)

rows = pd.concat(rows, ignore_index=True)

summary = []

for split in ["train", "test"]:
    df = rows[rows["split"] == split]
    y_pred = df["pred_score"].values
    y_true = df["key_frame"].values
    
    summary.append({
        "split": split,
        "num_samples": df.shape[0],
        "f1_score": f1_score(y_true, y_pred.round(), average='binary'),
        "precision": precision_score(y_true, y_pred.round(), average='binary'),
        "recall": recall_score(y_true, y_pred.round(), average='binary')
    })
    
summary = pd.DataFrame(summary)
summary

,split,num_samples,f1_score,precision,recall
0,train,141886,0.888474,0.893443,0.883560
1,test,61950,0.798160,0.824933,0.773069


In [25]:
rows = []

for sequence, df in results.items():
    split = "train" if sequence in train_sequences else "test" if sequence in test_sequences else "unknown"
    
    weather = sequence.split("_")[2]
    
    if len(weather) == 2:
        continue
    
    weather = weather.replace("Sunset", "").replace("Noon", "")
    
    if weather.endswith("Rain") or weather.endswith("Rainy"):
        weather = "Rain"
    
    town = sequence.split("_")[0]
    
    df.loc[:, "split"] = split
    df.loc[:, "weather"] = weather
    df.loc[:, "town"] = town
    
    rows.append(df)

rows = pd.concat(rows, ignore_index=True)

summary = []

for weather in rows["weather"].unique():
    for split in ["train", "test"]:
        df = rows[(rows["weather"] == weather) & (rows["split"] == split)]
        y_pred = df["pred_score"].values
        y_true = df["key_frame"].values
        
        summary.append({
            "weather": weather,
            "split": split,
            "num_samples": df.shape[0],
            "f1_score": f1_score(y_true, y_pred.round(), average='binary'),
            "precision": precision_score(y_true, y_pred.round(), average='binary'),
            "recall": recall_score(y_true, y_pred.round(), average='binary')
        })
        
summary = pd.DataFrame(summary)
summary

,weather,split,num_samples,f1_score,precision,recall
0,Rain,train,24975,0.876553,0.883827,0.869398
1,Rain,test,10989,0.823918,0.897492,0.761493
2,Clear,train,7992,0.874436,0.829881,0.924047
3,Clear,test,3996,0.788039,0.786998,0.789083
4,WetCloudy,train,8991,0.895512,0.881520,0.909956
5,WetCloudy,test,2997,0.857790,0.907674,0.813104
6,Cloudy,train,7992,0.833127,0.825518,0.840878
7,Cloudy,test,3996,0.754515,0.730902,0.779706
8,Wet,train,7992,0.904596,0.882881,0.927407
9,Wet,test,3996,0.826219,0.819751,0.832790


### Depth Extrapolation Model

In [26]:
def evaluate_metrics(pred, target, mask=None, deltas=(1.25, 1.25**2, 1.25**3), eps=1e-8):
    """
    Compute depth metrics for a single 2D depth image.

    Args:
        pred   : (H,W) predicted depth (numpy array)
        target : (H,W) ground truth depth (numpy array)
        mask   : (H,W) boolean or 0/1 array (optional).
                 If None, valid = target > 0.
        deltas : thresholds for δ accuracy
        eps    : small constant to avoid division/log issues

    Returns:
        dict with metrics: rmse, rmse_log, abs_rel, sq_rel, δ metrics
    """
    pred = pred[mask].clip(min=eps)
    target = target[mask].clip(min=eps)

    results = {}

    # δ-accuracy
    ratio = np.maximum(pred / target, target / pred)
    for d in deltas:
        results[f"δ<{d:.3f}"] = np.mean(ratio < d)

    diff = pred - target
    log_diff = np.log(pred) - np.log(target)
    
    # RMSE
    results["rmse"] = np.sqrt(np.mean(diff ** 2))

    # RMSE (log)
    results["rmse_log"] = np.sqrt(np.mean(log_diff ** 2))

    # AbsRel
    results["abs_rel"] = np.mean(np.abs(diff) / target)

    # SqRel
    results["sq_rel"] = np.mean((diff ** 2) / target)

    return results

In [ ]:
def evaluate_depth_estimation_dataset(hdf5_file, metadata_file, max_depth=200.0):
    metadata_df = pd.read_csv(metadata_file)
    results = []

    with h5py.File(hdf5_file, "r") as f:
        for i in tqdm.trange(metadata_df.shape[0]):
            split, sequence, _, prior_depth_frame_id, gt_depth_frame_id, duration = metadata_df.iloc[i]

            prior_depth_frame = f[f"{sequence}/depth_frames"][prior_depth_frame_id]
            gt_depth_frame = f[f"{sequence}/depth_frames"][gt_depth_frame_id]
            
            prior_depth_frame = prior_depth_frame
            gt_depth_frame = gt_depth_frame
            
            mask = gt_depth_frame < max_depth
            
            gt_depth_frame[gt_depth_frame > max_depth] = 0
            prior_depth_frame[prior_depth_frame > max_depth] = 0

            metrics_dict = evaluate_metrics(prior_depth_frame, gt_depth_frame, mask)
            
            results.append({
                "split": split,
                "duration": duration,
                "town": sequence.split("_")[0],
                **metrics_dict
            })
            
    return pd.DataFrame(results)

In [31]:
metadata_file = "data/extrapolation/variable_interval_voxel_grids_v1_metadata.csv"
hdf5_file = "data/extrapolation/variable_interval_voxel_grids_v1.h5"

results = evaluate_depth_estimation_dataset(hdf5_file, metadata_file, max_depth=200.0)
results.groupby("split").mean(numeric_only=True)

100%|██████████| 28140/28140 [03:20<00:00, 140.22it/s]


,duration,δ<1.250,δ<1.562,δ<1.953,rmse,rmse_log,abs_rel,sq_rel
split,,,,,,,,
test,34.280507,0.929521,0.953670,0.964744,8.414553,2.455032,0.072219,2.187817
train,34.043739,0.935519,0.955485,0.965772,8.271942,2.102627,0.070262,2.427828


In [35]:
results.groupby(["split", "duration"]).mean(numeric_only=True) # fps = 10000 / (duration * 5 milliseconds)

δ<1.250   δ<1.562   δ<1.953       rmse  rmse_log   abs_rel  \
split duration                                                                
test  4         0.975415  0.983564  0.987479   4.976189  1.486180  0.025566   
      10        0.957717  0.971196  0.977694   6.903771  2.015506  0.045201   
      20        0.941217  0.960450  0.969698   8.382200  2.373225  0.062560   
      40        0.909636  0.939895  0.954194   9.820024  2.980267  0.092261   
      100       0.860069  0.910972  0.932942  12.210876  3.486738  0.138993   
train 4         0.977296  0.983887  0.987404   5.024013  1.293441  0.026046   
      10        0.961197  0.972451  0.978530   6.701916  1.715878  0.044069   
      20        0.942551  0.959643  0.968805   8.220974  2.084743  0.064473   
      40        0.919456  0.944511  0.957273   9.623071  2.441528  0.087434   
      100       0.872510  0.913850  0.934513  12.117474  3.059070  0.134067   

                  sq_rel  
split duration            
test  4         0.696868  
      10        1.341285  
      20        1.884460  
      40        2.602588  
      100       4.520907  
train 4         0.928547  
      10        1.552863  
      20        2.257195  
      40        2.936012  
      100       4.626585

In [27]:
@torch.no_grad()
def evaluate_model(model: nn.Module, metadata_df: pd.DataFrame, hdf5_file: str, device: torch.device, max_depth: float = 200.0):
    model = model.half().eval()
    results = []
    
    df = metadata_df[metadata_df.split == "test"].reset_index(drop=True)
    
    with h5py.File(hdf5_file, "r") as f:
        for i in tqdm.trange(df.shape[0]):
            split, sequence, events_id, prior_depth_frame_id, gt_depth_frame_id, duration = df.iloc[i]

            depth_frame = f[f"{sequence}/depth_frames"][prior_depth_frame_id]
            event_frame = f[f"{sequence}/events"][events_id]
            gt_depth_frame = f[f"{sequence}/depth_frames"][gt_depth_frame_id]
            
            gt_mask = gt_depth_frame < max_depth
            
            gt_depth_frame[gt_depth_frame > max_depth] = 0
            depth_frame[depth_frame > max_depth] = 0

            depth_frame = torch.tensor(depth_frame, dtype=torch.float16, device=device).unsqueeze(0)
            event_frame = torch.tensor(event_frame, dtype=torch.float16, device=device).unsqueeze(0)
            
            pred_depth, _ = model(depth_frame, event_frame)
            pred_depth = pred_depth.detach().squeeze(0).cpu().numpy().astype(np.float32)
            
            metrics_dict = evaluate_metrics(pred_depth, gt_depth_frame, gt_mask)
            
            results.append({
                "split": split,
                "town": sequence.split("_")[0],
                "duration": duration,
                **metrics_dict
            })
            
    return pd.DataFrame(results)

In [36]:
model_dir = "checkpoints/SUNet2EnResConv_20250910_144511"

metadata_file = "data/extrapolation/variable_interval_voxel_grids_v1_metadata.csv"
hdf5_file = "data/extrapolation/variable_interval_voxel_grids_v1.h5"

metadata_df = pd.read_csv(metadata_file)
metadata_df = metadata_df[metadata_df.split == "test"].reset_index(drop=True)

config = TrainingConfig.load(os.path.join(model_dir, "config.pkl"))

model = SUNet2EnResConv(n_channels=16, activation=torch.nn.ReLU()).to(device)
model.load_state_dict(torch.load(os.path.join(model_dir, "best_model.pth"), map_location=device, weights_only=True))

results = evaluate_model(model, metadata_df, hdf5_file, device, max_depth=200.0)

results.groupby("split").mean(numeric_only=True)

100%|██████████| 5597/5597 [02:35<00:00, 35.98it/s]


,duration,δ<1.250,δ<1.562,δ<1.953,rmse,rmse_log,abs_rel,sq_rel
split,,,,,,,,
test,34.280507,0.941401,0.97292,0.984361,5.770404,0.325068,0.063458,0.910459


In [ ]:
results.groupby(["split", "duration"]).mean(numeric_only=True)